# 🏦 Retail Banking Product & Fees Assistant
### Hybrid RAG with Intent Routing, Grounded Citations & Refusal Mechanism

**Project:** Capstone — GenAI Banking Assistant  
**Stack:** FAISS + BM25 + RRF | Groq LLM | LLM-as-Judge Evaluator

---
**Pipeline Overview:**
```
Query → Intent Router → Filtered Retrieval (Dense + Sparse + RRF)
      → Confidence Check → Generator (with citations) → LLM-as-Judge Eval
```

## 📦 Section 1: Install Dependencies

In [1]:
!pip install -q faiss-cpu rank_bm25 sentence-transformers groq openai langchain langchain-community langchain-text-splitters tiktoken



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔑 Section 2: API Key Setup

In [2]:
import os
import time as _time
from functools import wraps
from dotenv import load_dotenv
load_dotenv()   # loads GROQ_API_KEY / OPENROUTER_API_KEY from .env

# ─────────────────────────────────────────────────────────────
# LLM Provider Config — change ONLY these two lines to switch
# ─────────────────────────────────────────────────────────────
LLM_PROVIDER = "groq"                       # "groq" | "openrouter"
LLM_MODEL    = "openai/gpt-oss-120b"        # Groq free-tier model (see your account's /models list)

# To switch to OpenRouter (minimal change):
#   LLM_PROVIDER = "openrouter"
#   LLM_MODEL = "meta-llama/llama-3.3-70b-instruct"   (free tier)
#   LLM_MODEL = "mistralai/mistral-7b-instruct"        (free tier)
# ─────────────────────────────────────────────────────────────

from openai import OpenAI   # openai SDK is compatible with both Groq + OpenRouter

if LLM_PROVIDER == "groq":
    client = OpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url="https://api.groq.com/openai/v1",
    )
    print(f"✅ Groq client initialized | model: {LLM_MODEL}")

elif LLM_PROVIDER == "openrouter":
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
        default_headers={"HTTP-Referer": "https://github.com/your-repo"},
    )
    print(f"✅ OpenRouter client initialized | model: {LLM_MODEL}")

else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

# ── Rate-limit retry wrapper (works for both providers) ──────
from openai import RateLimitError

_orig_create = client.chat.completions.create

@wraps(_orig_create)
def _create_with_retry(*args, **kwargs):
    max_retries = 8
    backoff = 1.5
    for attempt in range(max_retries):
        try:
            return _orig_create(*args, **kwargs)
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait = backoff * (attempt + 1)
            print(f"⚠️  Rate limit hit, retrying in {wait:.1f}s (attempt {attempt+2}/{max_retries})")
            _time.sleep(wait)

client.chat.completions.create = _create_with_retry
print("✅ Rate-limit retry wrapper enabled")


✅ Groq client initialized | model: openai/gpt-oss-120b
✅ Rate-limit retry wrapper enabled


## 📋 Section 4.5: Regenerate QA Set from Saved Docs

Regenerates evaluation Q&A pairs grounded in the **actual** documents in `data/raw/*.txt`
(so faithfulness, relevancy and refusal are all measurable and consistent with the KB).
The output is written to `qa_evaluation_set.txt`, which Section 14 later reads.

## 🔍 Section 5: Document Chunking & Ingestion

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class Chunk:
    chunk_id: str
    text: str
    doc_type: str   # 'product_terms' | 'fees' | 'eligibility'
    source_file: str

def load_and_chunk_documents() -> List[Chunk]:
    """
    Load raw docs and split into chunks with metadata.
    Chunks are split on markdown headers FIRST so every chunk keeps its
    product identity (e.g. 'NovaPrime') attached to the data rows.
    """
    sub_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=80,
        separators=["\n\n", "\n", ". ", " "]
    )
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
        strip_headers=False,
    )

    doc_config = [
        ("data/raw/product_terms.txt", "product_terms"),
        ("data/raw/fee_schedule.txt", "fees"),
        ("data/raw/eligibility_rules.txt", "eligibility")
    ]

    all_chunks = []
    for filepath, doc_type in doc_config:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        n_sec = 0
        for sec in header_splitter.split_text(text):
            n_sec += 1
            # Build a short context prefix from markdown headers, e.g.
            # '2. NovaPrime Savings Account – Premium Tier > 6.1 NovaCredit Card – Classic'
            prefix = " > ".join(v for _, v in sorted(sec.metadata.items()))
            body = sec.page_content.strip()
            if not body:
                continue
            if len(body) <= sub_splitter._chunk_size:
                pieces = [body]
            else:
                pieces = sub_splitter.split_text(body)
            for i, piece in enumerate(pieces):
                piece = piece.strip()
                if prefix and prefix not in piece:
                    piece = f"{prefix}\n{piece}"
                all_chunks.append(Chunk(
                    chunk_id=f"{doc_type}_{len(all_chunks):03d}",
                    text=piece,
                    doc_type=doc_type,
                    source_file=filepath
                ))
        print(f"✅ {doc_type}: {sum(1 for c in all_chunks if c.source_file == filepath)} chunks ({n_sec} sections)")

    return all_chunks

chunks = load_and_chunk_documents()
print(f"\n📦 Total chunks: {len(chunks)}")

d:\ML projects\Vibecode\ai-news-agent\ai-news-agent\venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ product_terms: 27 chunks (10 sections)
✅ fees: 20 chunks (9 sections)
✅ eligibility: 28 chunks (9 sections)

📦 Total chunks: 75


## 🧠 Section 6: Build Dense Index (FAISS + BGE Embeddings)

In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("⏳ Loading BGE embedding model...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("✅ Embedding model loaded")

def build_dense_index(chunks: List[Chunk]):
    """Encode chunks and build FAISS index."""
    texts = [c.text for c in chunks]
    print(f"⏳ Encoding {len(texts)} chunks...")

    # BGE requires a query prefix for retrieval
    embeddings = embed_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    ).astype(np.float32)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product (cosine after normalization)
    index.add(embeddings)

    print(f"✅ FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index, embeddings

faiss_index, chunk_embeddings = build_dense_index(chunks)

⏳ Loading BGE embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3451.23it/s]


✅ Embedding model loaded
⏳ Encoding 75 chunks...


Batches: 100%|██████████| 3/3 [00:05<00:00,  1.77s/it]

✅ FAISS index built: 75 vectors, dim=384


## 📝 Section 7: Build Sparse Index (BM25)

In [5]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> List[str]:
    """Simple tokenizer for BM25."""
    text = text.lower()
    tokens = re.findall(r'\b[a-z0-9]+\b', text)
    return tokens

corpus_tokens = [tokenize(c.text) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)

print(f"✅ BM25 index built over {len(corpus_tokens)} documents")

✅ BM25 index built over 75 documents


## 🔀 Section 8: Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [6]:
from typing import Optional

def dense_search(query: str, top_k: int = 20, doc_type_filter: Optional[str] = None) -> List[tuple]:
    """Dense retrieval with optional doc_type filter. Returns (chunk_idx, score) list."""
    # BGE query prefix
    query_vec = embed_model.encode(
        [f"Represent this sentence for searching relevant passages: {query}"],
        normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = faiss_index.search(query_vec, top_k * 3)  # over-fetch then filter
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx == -1:
            continue
        if doc_type_filter and chunks[idx].doc_type != doc_type_filter:
            continue
        results.append((idx, float(score)))
        if len(results) == top_k:
            break
    return results


def sparse_search(query: str, top_k: int = 20, doc_type_filter: Optional[str] = None) -> List[tuple]:
    """BM25 retrieval with optional doc_type filter. Returns (chunk_idx, score) list."""
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)

    # Get ranked indices
    ranked_indices = np.argsort(scores)[::-1]
    results = []
    for idx in ranked_indices:
        if doc_type_filter and chunks[idx].doc_type != doc_type_filter:
            continue
        results.append((idx, float(scores[idx])))
        if len(results) == top_k:
            break
    return results


def reciprocal_rank_fusion(dense_results: List[tuple], sparse_results: List[tuple],
                           k: int = 60, top_n: int = 5) -> List[tuple]:
    """
    RRF: score(d) = sum(1 / (k + rank_i))
    Returns top_n (chunk_idx, rrf_score) pairs.
    """
    rrf_scores: Dict[int, float] = {}

    for rank, (idx, _) in enumerate(dense_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    for rank, (idx, _) in enumerate(sparse_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:top_n]


PRODUCTS = [
    "novaprime", "novasavings", "nova fixed deposit", "novafd",
    "nova recurring deposit", "novard", "nova personal loan",
    "nova credit card", "nova creditcard", "nova bankk", "novabank"
]

def detect_product(query: str) -> Optional[str]:
    """Return the product name mentioned in the query, else None."""
    q = query.lower()
    for p in PRODUCTS:
        if p in q:
            return p
    return None

def promote_matching_product(fused: List[tuple], query: str) -> List[tuple]:
    """Stable-sort fused RRF results so chunks mentioning the queried product rank first.
    Fixes confusion between similar products (e.g. NovaSavings vs NovaPrime chunks).
    """
    product = detect_product(query)
    if product is None:
        return fused

    def rank_key(item):
        idx, _ = item
        return 0 if product in chunks[idx].text.lower() else 1

    return sorted(fused, key=rank_key)


def hybrid_retrieve(query: str, top_n: int = 5,
                    doc_type_filter: Optional[str] = None) -> List[Chunk]:
    """Full hybrid retrieval pipeline."""
    dense_res = dense_search(query, top_k=20, doc_type_filter=doc_type_filter)
    sparse_res = sparse_search(query, top_k=20, doc_type_filter=doc_type_filter)
    fused = reciprocal_rank_fusion(dense_res, sparse_res, top_n=top_n * 2)
    fused = promote_matching_product(fused, query)
    fused = fused[:top_n]
    return [chunks[idx] for idx, _ in fused], [score for _, score in fused]


# Quick test
test_chunks, test_scores = hybrid_retrieve("What is the minimum balance for savings account?")
print("\n🔍 Hybrid Retrieval Test:")
for chunk, score in zip(test_chunks, test_scores):
    print(f"  [{chunk.doc_type}] (rrf={score:.4f}) {chunk.text[:120]}...")


🔍 Hybrid Retrieval Test:
  [product_terms] (rrf=0.0325) ## 1. NovaSavings Account – Standard Savings  
| Feature | Details |
|---------|---------|
| **Base Interest Rate** | **...
  [eligibility] (rrf=0.0317) ## 2. NovaPrime Savings Account  
A premium‑tier savings product targeted at salaried professionals and high‑net‑worth i...
  [fees] (rrf=0.0315) ## 2️⃣ NovaPrime Savings Account  
| Fee / Service | Standard Charge | Waiver / Discount Conditions |
|---------------|-...
  [product_terms] (rrf=0.0309) 2. NovaPrime Savings Account – Premium Tier
| **Important T&C** | • Interest is posted monthly. <br>• Overdraft facility...
  [fees] (rrf=0.0305) ## 1️⃣ NovaSavings Account  
| Fee / Service | Standard Charge | Waiver / Discount Conditions |
|---------------|-------...


## 🎯 Section 9: Intent Router

Routes the query to the relevant document namespace before retrieval — reduces noise and improves precision.

In [7]:
import time
INTENT_SYSTEM_PROMPT = """
You are an intent classifier for a banking assistant.
Classify the user query into ONE of these intents:

- fees: Questions about charges, penalties, costs, service fees, transaction fees
- eligibility: Questions about who can apply, requirements, documents, criteria, qualifications
- product_terms: Questions about features, interest rates, tenure, limits, how products work
- out_of_scope: Questions unrelated to banking products (stocks, insurance, crypto, general advice)
- broad: Query spans multiple intents OR is ambiguous — search all document types

Respond with ONLY one word: fees | eligibility | product_terms | out_of_scope | broad
"""

# Intent → doc_type filter (None = search all namespaces)
INTENT_TO_DOC_TYPE = {
    "fees":          "fees",
    "eligibility":   "eligibility",
    "product_terms": "product_terms",
    "out_of_scope":  None,
    "broad":         None,   # FIX: was always defaulting wrong namespace
}

import re
VALID_INTENTS = ["fees", "eligibility", "product_terms", "out_of_scope", "broad"]

def parse_intent(raw: str) -> str:
    """Return whichever valid intent appears FIRST in the model reply."""
    text = raw.strip().lower().replace("out of scope", "out_of_scope")
    if not text:
        return "broad"
    best = (len(text) + 1, "broad")
    for token in VALID_INTENTS:
        m = re.search(rf"\b{re.escape(token)}\b", text)
        if m and m.start() < best[0]:
            best = (m.start(), token)
    return best[1]

def classify_intent(query: str) -> str:
    """
    Classify query intent. Returns: fees | eligibility | product_terms | out_of_scope | broad
    'broad' triggers unfiltered retrieval across all doc types.
    Retries only if the model reply had no intent label at all (empty/truncated output).
    """
    for attempt in range(3):
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": INTENT_SYSTEM_PROMPT},
                {"role": "user",   "content": query}
            ],
            temperature=0.0,
            max_tokens=512   # plenty of room; low caps (e.g. 20) cause empty/truncated replies
        )
        raw = response.choices[0].message.content
        cleaned = raw.strip().lower().replace("out of scope", "out_of_scope") if raw else ""
        if any(re.search(rf"\b{re.escape(t)}\b", cleaned) for t in VALID_INTENTS):
            return parse_intent(raw)
        if attempt < 2:
            print(f"  ⚠️  Empty/ambiguous reply (attempt {attempt+1}) → retrying")
    return "broad"

# ── Test ──────────────────────────────────────────────────────
test_queries = [
    "What is the NEFT charge for savings account?",
    "What is the minimum CIBIL score for personal loan?",
    "How does the fixed deposit interest work?",
    "What are the fee waiver eligibility criteria?",   # multi-intent → broad
    "Should I invest in Nifty 50 or Bitcoin?"
]

print("🎯 Intent Routing Tests:")
for q in test_queries:
    intent = classify_intent(q)
    print(f"  '{q[:60]}' → {intent} (filter: {INTENT_TO_DOC_TYPE[intent]})")
    time.sleep(0.5)


🎯 Intent Routing Tests:
  'What is the NEFT charge for savings account?' → fees (filter: fees)
  'What is the minimum CIBIL score for personal loan?' → eligibility (filter: eligibility)
  'How does the fixed deposit interest work?' → product_terms (filter: product_terms)
  'What are the fee waiver eligibility criteria?' → eligibility (filter: eligibility)
  'Should I invest in Nifty 50 or Bitcoin?' → out_of_scope (filter: None)


## 🛡️ Section 10: Confidence Check & Refusal Mechanism

Two-layer refusal:
1. **Intent-level**: out_of_scope → immediate refusal
2. **Retrieval-level**: top RRF score below threshold → low-confidence refusal

In [8]:
CONFIDENCE_THRESHOLD = 0.007  # RRF score threshold (tune based on your data)

def check_retrieval_confidence(scores: List[float]) -> bool:
    """Returns True if retrieval is confident enough to answer."""
    if not scores:
        return False
    return scores[0] >= CONFIDENCE_THRESHOLD


REFUSAL_MESSAGE = (
    "I'm sorry, I don't have reliable information to answer this question. "
    "Please contact your branch or our 24/7 helpline at 1800-XXX-XXXX for assistance."
)

OUT_OF_SCOPE_MESSAGE = (
    "This question is outside the scope of NovaBank's product and fee information. "
    "I can only assist with questions about our savings accounts, fixed deposits, "
    "personal loans, credit cards, fees, and eligibility criteria."
)

print("✅ Refusal mechanism configured")
print(f"   Confidence threshold: {CONFIDENCE_THRESHOLD}")

✅ Refusal mechanism configured
   Confidence threshold: 0.007


## 💬 Section 11: Generator with Grounded Citations

In [9]:
GENERATOR_SYSTEM_PROMPT = """
You are a strict and accurate banking assistant for NovaBank.
Answer using ONLY the provided context chunks — no outside knowledge.

STRICT RULES:
1. Every factual claim MUST be directly stated in context. Cite as [Source: chunk_id].
2. Do NOT infer, extrapolate, or generalize beyond the context.
3. If a specific value (rate, fee, limit) is absent from context, say:
   "This detail is not in our current documentation. Please contact your branch."
4. Never use 'typically', 'generally', 'usually' — only state what the context says.
5. Use bullet points for lists of fees or features.
6. If chunks conflict, cite both and flag the discrepancy.
"""

def generate_answer(query: str, retrieved_chunks: List[Chunk], top_n: int = 3) -> str:
    """
    Generate grounded answer from top_n chunks only.
    top_n=3 (not 5): fewer chunks = less hallucination surface.
    temp=0.0: maximum determinism for faithfulness.
    """
    context_str = "\n\n".join([
        f"[Source: {c.chunk_id}]\n{c.text}"
        for c in retrieved_chunks[:top_n]
    ])

    user_prompt = (
        f"Context (use ONLY this — no outside knowledge):\n{context_str}\n\n"
        f"Customer Question: {query}\n\n"
        f"Answer (cite every claim as [Source: chunk_id]):"
    )

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.0,   # deterministic = faithful
        max_tokens=600
    )
    return response.choices[0].message.content.strip()

print("✅ Generator configured (strict, top_n=3, temp=0.0)")


✅ Generator configured (strict, top_n=3, temp=0.0)


## 🔄 Section 12: Full RAG Pipeline

Putting it all together: Intent → Retrieve → Confidence Check → Generate

In [10]:
import time as time_module

def rag_pipeline(query: str, top_n: int = 5, verbose: bool = True) -> dict:
    """Full RAG pipeline: Intent → Retrieve → Confidence → Generate"""
    start = time_module.time()

    # Step 1: Intent Classification
    intent = classify_intent(query)
    if verbose:
        print(f"  🎯 Intent: {intent}")

    # Step 2: Out-of-scope refusal
    if intent == "out_of_scope":
        return {
            "query": query, "answer": OUT_OF_SCOPE_MESSAGE,
            "intent": intent, "retrieved_chunks": [], "rrf_scores": [],
            "refused": True, "refusal_reason": "out_of_scope",
            "latency_ms": round((time_module.time() - start) * 1000, 2)
        }

    # Step 3: Hybrid Retrieval with correct namespace filter
    doc_filter = INTENT_TO_DOC_TYPE.get(intent)   # None for broad/out_of_scope
    if verbose and doc_filter is None:
        print(f"  ℹ️  Broad retrieval (no namespace filter)")

    retrieved, scores = hybrid_retrieve(query, top_n=top_n, doc_type_filter=doc_filter)
    if verbose:
        print(f"  🔍 Retrieved {len(retrieved)} chunks (top RRF: {scores[0]:.4f})")

    # Step 4: Confidence check
    if not check_retrieval_confidence(scores):
        return {
            "query": query, "answer": REFUSAL_MESSAGE,
            "intent": intent, "retrieved_chunks": retrieved, "rrf_scores": scores,
            "refused": True, "refusal_reason": "low_confidence",
            "latency_ms": round((time_module.time() - start) * 1000, 2)
        }

    # Step 5: Generate — strict, top 3 chunks only
    answer = generate_answer(query, retrieved, top_n=3)

    return {
        "query": query, "answer": answer, "intent": intent,
        "retrieved_chunks": retrieved, "rrf_scores": scores,
        "refused": False, "refusal_reason": None,
        "latency_ms": round((time_module.time() - start) * 1000, 2)
    }


# ── Demo ──────────────────────────────────────────────────────
demo_queries = [
    "What are the NEFT charges for NovaSavings account?",
    "What are the fee waiver eligibility criteria for NovaSavings?",  # multi-intent → broad
    "Should I buy gold or invest in mutual funds?"                    # out of scope
]

for q in demo_queries:
    print(f"\n{'='*60}\n❓ Query: {q}")
    result = rag_pipeline(q, verbose=True)
    refused_str = f" ({result['refusal_reason']})" if result['refused'] else ""
    print(f"  🛡️ Refused: {result['refused']}{refused_str}")
    print(f"  ⏱️ Latency: {result['latency_ms']}ms")
    print(f"  💬 Answer:\n{result['answer']}")
    time.sleep(1)



❓ Query: What are the NEFT charges for NovaSavings account?
  🎯 Intent: fees
  🔍 Retrieved 5 chunks (top RRF: 0.0328)
  🛡️ Refused: False
  ⏱️ Latency: 1331.99ms
  💬 Answer:
- **NEFT Transfer (outward) – NovaSavings Account**  
  - Up to 5 transactions are free each month.  
  - After the free quota, each transaction costs **₹2.50**.  
  - The fee is waived for customers using the online‑banking “no‑fee” tier.  

[Source: fees_029]

❓ Query: What are the fee waiver eligibility criteria for NovaSavings?
  🎯 Intent: eligibility
  🔍 Retrieved 5 chunks (top RRF: 0.0325)
  🛡️ Refused: False
  ⏱️ Latency: 1822.23ms
  💬 Answer:
This detail is not in our current documentation. Please contact your branch.

❓ Query: Should I buy gold or invest in mutual funds?
  🎯 Intent: out_of_scope
  🛡️ Refused: True (out_of_scope)
  ⏱️ Latency: 437.35ms
  💬 Answer:
This question is outside the scope of NovaBank's product and fee information. I can only assist with questions about our savings accounts, fixed

## ⚖️ Section 13: LLM-as-Judge Evaluator

Evaluates three metrics:
- **Faithfulness** — is the answer grounded in the retrieved context?
- **Answer Relevancy** — does the answer address the question?
- **Refusal Correctness** — did the system refuse when it should have (and not refuse when it shouldn't)?

In [11]:
FAITHFULNESS_PROMPT = """
You are evaluating whether an AI answer is faithful to the provided context.

Context:
{context}

Question: {question}
Answer: {answer}

Evaluate faithfulness: Does the answer ONLY contain claims supported by the context?
Score 0.0 to 1.0 where:
- 1.0 = all claims are grounded in context
- 0.5 = some claims unsupported
- 0.0 = answer contradicts or ignores context

Respond ONLY with a JSON object: {{"score": <float>, "reason": "<one sentence>"}}
"""

RELEVANCY_PROMPT = """
You are evaluating whether an AI answer addresses the user question.

Question: {question}
Answer: {answer}

Score 0.0 to 1.0 where:
- 1.0 = directly and completely answers the question
- 0.5 = partially answers or goes off-topic
- 0.0 = completely misses the question

Respond ONLY with a JSON object: {{"score": <float>, "reason": "<one sentence>"}}
"""


import re as _re

def extract_json(raw: str) -> dict:
    """Robustly pull a valid JSON object out of the model reply."""
    if not raw:
        return {}
    # Strip markdown fences and leading prose, keep the first balanced {...} block.
    s = raw.strip()
    for _ in range(5):
        try:
            return json.loads(s)
        except Exception:
            pass
        # Drop any leading ```/```json fence markers
        if s.startswith("```"):
            s = s[3:].lstrip()
            if s.startswith("json"):
                s = s[4:].lstrip()
            continue
        # Otherwise keep the first {...} ... last } substring
        start, end = s.find("{"), s.rfind("}")
        if start == -1 or end < start:
            return {}
        s = s[start:end + 1]
    try:
        return json.loads(s)
    except Exception:
        return {}

def llm_judge(prompt_template: str, **kwargs) -> dict:
    prompt = prompt_template.format(**kwargs)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,   # enough for full JSON; low caps truncate mid-JSON -> parse error
        # response_format={"type": "json_object"},   # Removed: unsupported by some Groq models, extract_json handles it
    )
    raw = response.choices[0].message.content
    parsed = extract_json(raw)
    if not parsed:
        print(f"[DEBUG] llm_judge raw response: {repr(raw)}")
        return {"score": 0.0, "reason": "parse error"}
    return parsed


def evaluate_result(result: dict, ground_truth_answer: str = None,
                    should_be_refused: bool = False) -> dict:
    metrics = {}

    # Refusal correctness
    metrics["refusal_correctness"] = (
        1.0 if (should_be_refused == result["refused"]) else 0.0
    )

    if result["refused"]:
        metrics["faithfulness"] = None
        metrics["relevancy"] = None
        return metrics

    context = "".join([c.text for c in result["retrieved_chunks"]])

    faith = llm_judge(FAITHFULNESS_PROMPT, context=context,
                      question=result["query"], answer=result["answer"])
    metrics["faithfulness"] = faith.get("score", 0.0)
    metrics["faithfulness_reason"] = faith.get("reason", "")
    time.sleep(0.5)

    rel = llm_judge(RELEVANCY_PROMPT, question=result["query"], answer=result["answer"])
    metrics["relevancy"] = rel.get("score", 0.0)
    metrics["relevancy_reason"] = rel.get("reason", "")

    return metrics


print("✅ LLM-as-Judge evaluator configured")


✅ LLM-as-Judge evaluator configured


## 📈 Section 14: Full Evaluation Run on QA Set

In [18]:
# Load QA set
import re

qa_set = []
with open("qa_evaluation_set.txt", "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        m = re.match(r"^(?:\d+\.\s*)?Q:\s*(.*?)\s*\|\s*A:\s*(.*)$", line)
        if m:
            qa_set.append({"question": m.group(1), "answer": m.group(2)})

# Run evaluation on a sample (run 10 queries for now)
EVAL_SAMPLE_SIZE = 5
eval_sample = qa_set[:EVAL_SAMPLE_SIZE]

print(f"\n📊 Running evaluation on {EVAL_SAMPLE_SIZE} samples...")
print("(This will take a few minutes due to API calls)\n")

eval_results = []
latencies = []

for i, qa in enumerate(eval_sample):
    print(f"  [{i+1}/{EVAL_SAMPLE_SIZE}] {qa['question'][:200]}...")

    # Run pipeline
    result = rag_pipeline(qa["question"], verbose=False)
    latencies.append(result["latency_ms"])

    # Evaluate
    should_refuse = not qa.get("answerable", True)
    metrics = evaluate_result(result, should_be_refused=should_refuse)

    eval_results.append({
        "question": qa["question"],
        "expected_answer": qa.get("answer", ""),
        "generated_answer": result["answer"],
        "intent": result["intent"],
        "refused": result["refused"],
        "should_refuse": should_refuse,
        "latency_ms": result["latency_ms"],
        **metrics
    })

    time.sleep(2)  # Rate limit buffer

print("\n✅ Evaluation complete!")


📊 Running evaluation on 5 samples...
(This will take a few minutes due to API calls)

  [1/5] What is the base interest rate for the NovaSavings Account on the first ₹2 lakh of average daily balance?...
  [2/5] What fee is charged if I close my NovaSavings account within 30 days of opening?...
  [3/5] What is the maximum daily withdrawal limit for the NovaPrime Savings Account?...
  [4/5] What is the interest rate on the overdraft facility for the NovaSavings Account?...
  [5/5] What is the minimum balance requirement for an individual opening a NovaPrime Savings Account?...

✅ Evaluation complete!


## 📊 Section 15: Metrics Summary

In [19]:
import statistics
import json

# Filter answered vs refused
answered = [r for r in eval_results if not r["refused"]]
refused_results = [r for r in eval_results if r["refused"]]

# Faithfulness
faith_scores = [r["faithfulness"] for r in answered if r.get("faithfulness") is not None]
avg_faithfulness = statistics.mean(faith_scores) if faith_scores else 0

# Relevancy
rel_scores = [r["relevancy"] for r in answered if r.get("relevancy") is not None]
avg_relevancy = statistics.mean(rel_scores) if rel_scores else 0

# Refusal correctness
refusal_correct = [r for r in eval_results if r["refusal_correctness"] == 1.0]
refusal_correctness = len(refusal_correct) / len(eval_results) if eval_results else 0

# Latency
sorted_latencies = sorted(latencies)
p95_latency = sorted_latencies[int(len(sorted_latencies) * 0.95)] if sorted_latencies else 0
avg_latency = statistics.mean(latencies) if latencies else 0

print("\n" + "="*50)
print("📊 EVALUATION RESULTS")
print("="*50)
print(f"  Samples evaluated    : {len(eval_results)}")
print(f"  Answered             : {len(answered)}")
print(f"  Refused              : {len(refused_results)}")
print(f"")
print(f"  Faithfulness         : {avg_faithfulness:.3f}")
print(f"  Answer Relevancy     : {avg_relevancy:.3f}")
print(f"  Refusal Correctness  : {refusal_correctness:.3f}")
print(f"")
print(f"  Avg Latency          : {avg_latency:.0f}ms")
print(f"  p95 Latency          : {p95_latency:.0f}ms")
print("="*50)

# Save results
with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2, default=str)

summary = {
    "samples": len(eval_results),
    "faithfulness": round(avg_faithfulness, 3),
    "answer_relevancy": round(avg_relevancy, 3),
    "refusal_correctness": round(refusal_correctness, 3),
    "avg_latency_ms": round(avg_latency, 1),
    "p95_latency_ms": round(p95_latency, 1)
}
with open("summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Results saved to data/eval/")


📊 EVALUATION RESULTS
  Samples evaluated    : 5
  Answered             : 5
  Refused              : 0

  Faithfulness         : 1.000
  Answer Relevancy     : 0.800
  Refusal Correctness  : 1.000

  Avg Latency          : 1493ms
  p95 Latency          : 2461ms

✅ Results saved to data/eval/


## 🔬 Section 16: Error Analysis

In [20]:
# Low faithfulness cases
low_faith = [
    r for r in answered
    if r.get("faithfulness") is not None and r["faithfulness"] < 0.7
]

# Wrong refusals
wrong_refusals = [r for r in eval_results if r["refusal_correctness"] == 0.0]

print(f"⚠️  Low Faithfulness Cases (< 0.7): {len(low_faith)}")
for r in low_faith[:3]:
    print(f"  Q: {r['question'][:80]}")
    print(f"  Faith: {r['faithfulness']:.2f} | {r.get('faithfulness_reason', '')}")
    print()

print(f"\n⚠️  Wrong Refusal Decisions: {len(wrong_refusals)}")
for r in wrong_refusals[:3]:
    refused_when = "should NOT have" if not r["should_refuse"] else "should have"
    print(f"  Q: {r['question'][:80]}")
    print(f"  System refused={r['refused']} but {refused_when} refused")
    print()

⚠️  Low Faithfulness Cases (< 0.7): 0

⚠️  Wrong Refusal Decisions: 0


## 🎮 Section 17: Interactive Demo

In [21]:
def interactive_query(query: str):
    """Pretty-print a single query through the full pipeline."""
    print("\n" + "="*60)
    print(f"❓ Query: {query}")
    print("-"*60)

    result = rag_pipeline(query, verbose=True)

    print(f"\n💬 Answer:")
    print(result["answer"])

    if result["retrieved_chunks"] and not result["refused"]:
        print(f"\n📎 Sources used:")
        for chunk in result["retrieved_chunks"]:
            print(f"  • [{chunk.chunk_id}] ({chunk.doc_type}): {chunk.text[:80]}...")

    print(f"\n⏱️ Latency: {result['latency_ms']}ms")
    print("="*60)


# Try your own queries!
interactive_query("What documents do I need to apply for a NovaPersonal Loan?")


❓ Query: What documents do I need to apply for a NovaPersonal Loan?
------------------------------------------------------------
  🎯 Intent: eligibility
  🔍 Retrieved 5 chunks (top RRF: 0.0320)

💬 Answer:
**Documents required for a NovaPersonal Loan**

- **For Salaried applicants**  
  - PAN Card ✔ [Source: eligibility_065]  
  - Aadhaar Card ✔ [Source: eligibility_065]  
  - Proof of Address ✔ [Source: eligibility_065]  
  - Salary Slips (last 3

📎 Sources used:
  • [eligibility_065] (eligibility): 5. NovaPersonal Loan
**Documents Required**  
| Document | Salaried | Self‑Emplo...
  • [eligibility_064] (eligibility): 5. NovaPersonal Loan
| **Maximum Loan Amount** | Up to **₹25Lakhs** for salaried...
  • [eligibility_066] (eligibility): 5. NovaPersonal Loan
| GST Returns (last 2 quarters) | ✖ | ✔ |
| Business Proof ...
  • [eligibility_063] (eligibility): 5. NovaPersonal Loan
| **Credit Score (CIBIL)** | Minimum **750** for standard p...
  • [eligibility_062] (eligibility): ## 5. Nova

In [22]:
# Try more queries
interactive_query("What is the annual fee for NovaCreditCard Platinum?")


❓ Query: What is the annual fee for NovaCreditCard Platinum?
------------------------------------------------------------
  🎯 Intent: fees
  🔍 Retrieved 5 chunks (top RRF: 0.0328)

💬 Answer:
- **Annual Fee:** ₹5,000 (waived if you spend ≥ ₹5 Lakh in a calendar year) [Source: fees_040]

📎 Sources used:
  • [fees_040] (fees): ## 6️⃣ NovaCreditCard Platinum (Premium Tier)  
| Fee / Service | Standard Charg...
  • [fees_038] (fees): ## 5️⃣ NovaCreditCard Classic  
| Fee / Service | Standard Charge |
|-----------...
  • [fees_041] (fees): 6️⃣ NovaCreditCard Platinum (Premium Tier)
| **Over‑limit Fee** | **₹750** per o...
  • [fees_042] (fees): 6️⃣ NovaCreditCard Platinum (Premium Tier)
| **Card Replacement (lost/stolen)** ...
  • [fees_039] (fees): 5️⃣ NovaCreditCard Classic
| **Cash Advance Fee** | **2%** of cash‑advance amoun...

⏱️ Latency: 1245.95ms


In [23]:
interactive_query("How do I invest in the stock market?")  # Should be refused


❓ Query: How do I invest in the stock market?
------------------------------------------------------------
  🎯 Intent: out_of_scope

💬 Answer:
This question is outside the scope of NovaBank's product and fee information. I can only assist with questions about our savings accounts, fixed deposits, personal loans, credit cards, fees, and eligibility criteria.

⏱️ Latency: 678.9ms
